# data_preparation.ipynb
## Pipeline de consolidación del dataset acústico
### TFG — Análisis de biomarcadores acústicos en ELA

**Autor:** Jakub Wysocki  

---

Este notebook genera el dataset acústico fusionado a partir de los 5 archivos
Excel de MATLAB (uno por vocal). El resultado es `dataset_raw_fusionado.csv`
con 68 filas × 204 columnas (4 metadatos + 200 features acústicas).

**Nota:** Las etiquetas clínicas y el filtrado de los 5 sujetos excluidos
se realizan en el siguiente notebook: `label_integration.ipynb`.

### Estructura de ejecución:
1. Configuración del entorno
2. Ejecución del pipeline de preprocesamiento
3. Inspección del dataset fusionado
4. Verificación de integridad

## 0. Configuración del entorno

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Añadir raíz del proyecto al path para importar src/
PROJECT_ROOT = Path('..').resolve()   # TFG_project/
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    run_preprocessing_pipeline,
    validate_dataset,
    EXPECTED_ROWS_RAW,
    EXPECTED_SUBJECTS_FINAL,
)

RAW_DIR       = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

print(f'Raíz del proyecto : {PROJECT_ROOT}')
print(f'Datos brutos      : {RAW_DIR}')
print(f'Datos procesados  : {PROCESSED_DIR}')
print(f'Filas esperadas (raw)   : {EXPECTED_ROWS_RAW}   (50 ELA + 18 controles)')
print(f'Filas esperadas (final) : {EXPECTED_SUBJECTS_FINAL} (45 ELA + 18 controles)')

## 1. Pipeline de preprocesamiento

El pipeline ejecuta los siguientes pasos automáticamente:

| Paso | Operación |
|------|-----------|
| 1 | Carga de los 5 archivos `.xls` (vocal a/e/i/o/u) |
| 2 | Validación y unificación de la columna `Genero` |
| 3 | Fusión horizontal con sufijo de vocal (`_a`, `_e`, `_i`, `_o`, `_u`) |
| 4 | Construcción del dataset con metadatos placeholder |
| 5 | Validación de integridad y guardado |

In [ ]:
df = run_preprocessing_pipeline(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    output_filename='dataset_raw_fusionado.csv',
)

print(f'\nDataset fusionado generado: {df.shape[0]} filas × {df.shape[1]} columnas')

## 2. Inspección del dataset fusionado

In [ ]:
# Estructura general
meta_cols    = ['subject_id', 'genero', 'label_clinico', 'label_maquina']
feature_cols = [c for c in df.columns if c not in meta_cols]
features_a   = [c for c in feature_cols if c.endswith('_a')]

print('=== ESTRUCTURA DEL DATASET ===')
print(f'  Filas totales     : {df.shape[0]}  (68 = 50 HUB + 18 K)')
print(f'  Columnas totales  : {df.shape[1]}')
print(f'  Metadatos         : {len(meta_cols)} {meta_cols}')
print(f'  Features acústicas: {len(feature_cols)} (5 vocales × 40 biomarcadores)')
print(f'  Features vocal \'a\': {len(features_a)}')

In [ ]:
# Primeras filas — columnas de metadatos
# subject_id es provisional (S001...S068), se reemplaza en label_integration.ipynb
print('=== METADATOS (primeras 10 filas) ===')
print('Nota: subject_id provisional S001...S068 → se reemplaza por HUB/K en el siguiente notebook')
df[meta_cols].head(10)

In [ ]:
# Distribución de género
genero_map = {0: 'Mujer', 1: 'Hombre'}
print('=== DISTRIBUCIÓN DE GÉNERO ===')
print(df['genero'].map(genero_map).value_counts().to_string())
print(f'Total: {len(df)} sujetos')

In [ ]:
# Estadísticas descriptivas — primeras 8 features de vocal 'a'
print('=== ESTADÍSTICAS DESCRIPTIVAS (vocal \'a\', primeras 8 features) ===')
df[features_a[:8]].describe().round(4)

## 3. Verificación de integridad

In [ ]:
# Nulos en features acústicas
null_counts   = df[feature_cols].isnull().sum()
null_nonzero  = null_counts[null_counts > 0]

if null_nonzero.empty:
    print('✓ Sin valores nulos en features acústicas.')
else:
    print(f'⚠ Features con nulos ({len(null_nonzero)} columnas):')
    print(null_nonzero)

In [ ]:
# Duplicados
n_dup = df.duplicated().sum()
print(f'Filas duplicadas: {n_dup}')
print('✓ Sin duplicados.' if n_dup == 0 else f'⚠ {n_dup} duplicados encontrados.')

In [ ]:
# Verificar CSV guardado
csv_path = PROCESSED_DIR / 'dataset_raw_fusionado.csv'
df_check = pd.read_csv(csv_path)

print('=== VERIFICACIÓN CSV GUARDADO ===')
print(f'  Archivo : {csv_path}')
print(f'  Shape   : {df_check.shape}  → esperado (68, 204)')
print(f'  Nulos   : {df_check.isnull().sum().sum()}')
assert df_check.shape == (68, 204), (
    f'Shape inesperado: {df_check.shape}. Revisa los archivos .xls en data/raw/'
)
print('\n✓ dataset_raw_fusionado.csv correcto.')

---
## Resumen

| Paso | Resultado | Estado |
|------|-----------|--------|
| Carga 5 archivos .xls | 68 sujetos × 41 cols/vocal | ✓ |
| Unificación `Genero` | 1 columna canónica | ✓ |
| Fusión vocales | 200 features acústicas | ✓ |
| Dataset fusionado | 68 × 204 | ✓ |
| Etiquetas | NaN (pendiente) | ⏳ |

**Siguiente notebook:** `label_integration.ipynb`  
→ Asigna etiquetas clínicas y elimina los 5 sujetos excluidos.  
→ Genera `dataset_final.csv` con 63 filas y etiquetas completas.